# Phase 3a — Optimal Cluster Count Selection
## FCM Gradational Lithofacies Classification
**Author:** Kumar Yuvraj (23GG5PE02) | IIT Kharagpur

FCM requires the number of clusters `c` to be specified in advance — unlike DBSCAN.  
We evaluate `c = 2…15` using three complementary metrics:

| Metric | Direction | What it measures |
|--------|-----------|------------------|
| Partition Coefficient (PC / FPC) | ↑ higher = better | How 'crisp' is the partition — are memberships concentrated? |
| Fuzzy Partition Entropy (FPE) | ↓ lower = better | Uncertainty / spread of membership assignments |
| Within-cluster SS (WCSS) | elbow = optimal | Inertia — how compact are the clusters? |

**Optimal c** is at the joint elbow: PC stops rising, FPE stops falling.

In [4]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skfuzzy as fuzz
os.chdir('D:/Projects/fcm-lithofacies')
warnings.filterwarnings('ignore')

OUT_DATA = './outputs/data/'
OUT_FIGS = './outputs/figures/'

# FCM parameters — do not change m=2.0 without strong justification
# m=2 is the industry-standard starting value (Bezdek, 1981)
M_FUZZ   = 2.0
ERROR    = 0.005
MAXITER  = 1000
C_RANGE  = range(2, 16)   # test c = 2 … 15

# Load scaled features
X_scaled = np.load(os.path.join(OUT_DATA, 'X_scaled.npy'))
print(f'X_scaled shape: {X_scaled.shape}')

# ── Sub-sample for speed (skfuzzy is O(N·C·iter)) ─────────────────────────
# We use 80,000 rows for cluster selection — enough for stable PC/FPE curves
# while keeping runtime to ~5 minutes.  Final model (Phase 3b) uses full data.
rng = np.random.default_rng(42)
SUBSAMPLE = 80_000
if X_scaled.shape[0] > SUBSAMPLE:
    idx = rng.choice(X_scaled.shape[0], SUBSAMPLE, replace=False)
    X_sub = X_scaled[idx]
else:
    X_sub = X_scaled

print(f'Using {len(X_sub):,} rows for cluster selection scan')

X_scaled shape: (1127735, 5)
Using 80,000 rows for cluster selection scan


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Run FCM for c = 2 to 15, record PC, FPE, WCSS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY record all three metrics:
# PC and FPE are FCM-specific; WCSS is the K-Means equivalent ('elbow method').
# Using all three guards against a degenerate solution where one metric
# is misleading (e.g., PC keeps rising monotonically but the elbow is clear
# in WCSS).  Convergence of all three around the same c is the strongest signal.

X_input = X_sub.T   # skfuzzy expects (n_features, n_samples)

results = []

for c in C_RANGE:
    np.random.seed(42)   # reproducibility
    cntr, u, _, d, jm, _, fpc = fuzz.cluster.cmeans(
        X_input,
        c=c,
        m=M_FUZZ,
        error=ERROR,
        maxiter=MAXITER,
        init=None,
    )
    # PC / FPC — returned directly by skfuzzy
    pc = fpc

    # Fuzzy Partition Entropy (FPE) — must compute manually
    # FPE = -(1/N) * Σ_i Σ_j u_ij * log(u_ij)
    u_safe = np.clip(u, 1e-10, 1.0)
    fpe = -float(np.mean(np.sum(u_safe * np.log(u_safe), axis=0)))

    # WCSS — sum of squared distances from each point to its winner centroid
    # d has shape (c, n_samples) — squared distances
    hard = np.argmax(u, axis=0)           # winner cluster per sample
    wcss = float(np.sum(
        d[hard, np.arange(d.shape[1])]    # distance to winner centroid
    ))

    results.append({'c': c, 'PC': pc, 'FPE': fpe, 'WCSS': wcss})
    print(f'  c={c:2d}   PC={pc:.4f}   FPE={fpe:.4f}   WCSS={wcss:.2e}')

df_results = pd.DataFrame(results)
df_results

  c= 2   PC=0.7000   FPE=0.4655   WCSS=1.23e+05
  c= 3   PC=0.5228   FPE=0.8129   WCSS=1.13e+05
  c= 4   PC=0.4177   FPE=1.0737   WCSS=1.08e+05
  c= 5   PC=0.3631   FPE=1.2484   WCSS=1.03e+05
  c= 6   PC=0.3213   FPE=1.4065   WCSS=9.84e+04
  c= 7   PC=0.2844   FPE=1.5489   WCSS=9.57e+04
  c= 8   PC=0.2568   FPE=1.6706   WCSS=9.37e+04
  c= 9   PC=0.2340   FPE=1.7788   WCSS=9.21e+04
  c=10   PC=0.2161   FPE=1.8736   WCSS=9.03e+04
  c=11   PC=0.1997   FPE=1.9645   WCSS=8.93e+04
  c=12   PC=0.1857   FPE=2.0469   WCSS=8.80e+04
  c=13   PC=0.1745   FPE=2.1201   WCSS=8.70e+04
  c=14   PC=0.1646   FPE=2.1894   WCSS=8.62e+04
  c=15   PC=0.1549   FPE=2.2561   WCSS=8.54e+04


,c,PC,FPE,WCSS
0,2,0.700009,0.465457,123378.081797
1,3,0.522819,0.812897,113250.209723
2,4,0.417656,1.073651,107725.169941
3,5,0.363135,1.248414,102755.496529
4,6,0.321264,1.406458,98385.692726
5,7,0.284405,1.548939,95737.493195
6,8,0.256765,1.670618,93738.174700
7,9,0.233966,1.778835,92065.987017
8,10,0.216115,1.873579,90344.994155
9,11,0.199651,1.964534,89339.983229


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Plot all three metrics; mark optimal c
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WHY use second-difference to find the elbow automatically:
# The elbow is the point of maximum curvature.  The second difference of
# a monotone curve peaks at the inflection point — equivalent to the
# geometric elbow.  We apply it to PC (first local maximum in second diff
# = where PC starts flattening).

# WHY override auto-detection?
# The PC curve for FORCE 2020 logs is monotonically decreasing (0.70 → 0.15)
# with no clear elbow — this is common when the feature space has a continuous
# gradational structure rather than tight discrete blobs.  The second-difference
# elbow detector fires at c=3 which is geologically too coarse to separate
# sandstone, shale, coal, and carbonate end-members.
#
# We override to C=7 based on domain knowledge:
#   FORCE 2020 has 6 dominant petrophysical end-members:
#   clean sand, silty sand, sandy shale, shale, coal, carbonate.
#   C=7 gives one extra cluster to absorb marl/transitional facies
#   without overfitting to individual wells (C=8+ risk).
C_OPTIMAL = 7   # domain-knowledge override — do not change without re-reading centroid table

print(f'→ C_OPTIMAL set to: c = {C_OPTIMAL}  (manual override)')
print(f'  Auto-detector suggested c=3 but that is too coarse for this dataset.')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f'FCM Cluster Selection Metrics  (m={M_FUZZ}, subsample={len(X_sub):,})',
    fontsize=12, fontweight='bold'
)

metrics = [
    ('PC',   'Partition Coefficient (↑ better)',   'steelblue'),
    ('FPE',  'Fuzzy Partition Entropy (↓ better)', 'darkorange'),
    ('WCSS', 'WCSS / Inertia (elbow)',             'seagreen'),
]

for ax, (col, title, colour) in zip(axes, metrics):
    ax.plot(df_results['c'], df_results[col], 'o-',
            color=colour, lw=2, ms=7)
    ax.axvline(C_OPTIMAL, color='crimson', ls='--', lw=1.8,
               label=f'Optimal c = {C_OPTIMAL}')
    ax.set_xlabel('Number of clusters  c', fontsize=10)
    ax.set_ylabel(col, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.35)
    ax.set_xticks(list(C_RANGE))

plt.tight_layout()
fig_path = os.path.join(OUT_FIGS, 'cluster_selection_metrics.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f'\n✓ Saved: {fig_path}')

# Save optimal c for next notebook
import json
with open(os.path.join(OUT_DATA, 'cluster_selection.json'), 'w') as f:
    json.dump({'C_OPTIMAL': C_OPTIMAL, 'M_FUZZ': M_FUZZ,
               'metrics': df_results.to_dict(orient='records')}, f, indent=2)
print(f'✓ Saved: cluster_selection.json  (C_OPTIMAL = {C_OPTIMAL})')

→ C_OPTIMAL set to: c = 7  (manual override)
  Auto-detector suggested c=3 but that is too coarse for this dataset.

✓ Saved: ./outputs/figures/cluster_selection_metrics.png
✓ Saved: cluster_selection.json  (C_OPTIMAL = 7)


## Geological Interpretation of Optimal c

The optimal cluster count from the PC/FPE elbow will typically fall between **5 and 9** for the FORCE 2020 dataset — well below the 12 hard lithofacies labels in the competition ground truth. This is not a failure of FCM; it is a feature.

**Why fewer clusters than hard labels?**  
FCM absorbs transition zones into the *probability space* rather than creating separate discrete classes for each facies variant.  A depth interval that is 60% sandstone and 40% sandy-shale does not need its own cluster — its geological nature is fully encoded in its membership vector.  This collapses what would otherwise be multiple 'transition' classes (Sandstone, Sandstone/Shale, silty-sand) into 2–3 clusters with varying membership weights.

**Geological meaning of each cluster (provisional — confirm after Phase 3b centroid analysis):**

| Cluster (likely) | GR | RHOB | NPHI | Interpretation |
|------------------|----|------|------|----------------|
| Clean sand | Low | Moderate-high | Low | Barakar-type fluvial channel sandstone |
| Silty sand | Low-moderate | Moderate | Low-moderate | Transition / crevasse splay |
| Sandy shale | Moderate-high | Low-moderate | High | Overbank / levee |
| Shale | High | Low | High | Floodplain shale |
| Coal | Very low | Very low | High | Mire / coal swamp |
| Carbonate | Low-moderate | High | Low | Dolomite / limestone facies |

Verify and label after seeing actual centroid values in Phase 3b.